In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_8.py ──
"""
Shared infrastructure for MLFP03 Exercise 8 — Production ML + Drift +
Deployment.

Contains: data loading (Singapore credit scoring via MLFPDataLoader) with
decoded protected-attribute groups, the baseline model (TrainingPipeline +
engine isotonic calibration, registered in this exercise's registry),
split-conformal helpers, per-group fairness measurement, PSI/KS drift
helpers, and the common output directory.

Technique-specific code (conformal quantile logic, dashboard rendering,
readiness checklist) stays in the per-technique files.
"""

import asyncio
import os
import pickle
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from scipy import stats
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from kailash_ml.interop import to_sklearn_input
from kailash_ml.types import FeatureField, FeatureSchema


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex8_production"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring (from MLFP02)
# ════════════════════════════════════════════════════════════════════════

RANDOM_SEED = 42

# Columns that MUST NOT be model inputs (Lesson 3.1 leakage rule):
#   customer_id              — a row identifier, not a property of the applicant
#   future_default_indicator — recorded AFTER the loan outcome is known; it
#                              agrees with ``default`` on ~99% of rows, so a
#                              model that sees it "predicts" default by
#                              reading the answer (Exercise 4 screens for it).
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = ("customer_id", "future_default_indicator")


def _decode_groups(
    X: np.ndarray, feature_names: list[str], mappings: dict[str, dict[str, int]]
) -> dict[str, np.ndarray]:
    """Human-readable protected-attribute labels for each row of ``X``.

    ``race`` and ``gender`` are decoded from the pipeline's ordinal codes;
    ``age`` is banded. These are the groups a fairness audit reports on.
    """
    groups: dict[str, np.ndarray] = {}
    for col in ("race", "gender"):
        inverse = {code: label for label, code in mappings[col].items()}
        codes = X[:, feature_names.index(col)].astype(int)
        groups[col] = np.array([inverse.get(c, "unknown") for c in codes])
    age = X[:, feature_names.index("age")]
    groups["age_band"] = np.select(
        [age < 30, age < 45, age < 60], ["<30", "30-44", "45-59"], default="60+"
    )
    return groups


def load_credit_split() -> dict[str, Any]:
    """Load Singapore credit scoring data, preprocess, and return a split.

    Returns a dict with keys:
        X_train, y_train, X_test, y_test : numpy arrays
        feature_names                    : list[str]
        default_rate                     : float (train positive rate)
        test_groups                      : {"race"|"gender"|"age_band": labels}
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    # Drop the row identifier and the post-outcome leak column BEFORE
    # preprocessing — see CREDIT_NON_FEATURE_COLUMNS above.
    credit = credit.drop(CREDIT_NON_FEATURE_COLUMNS)

    # Split FIRST, then fit imputation/encoding on the training rows only:
    # PreprocessingPipeline.setup() on the whole frame would fit them on the
    # test rows too (it splits only after fitting).
    result = split_then_preprocess(
        credit,
        target="default",
        test_size=0.2,
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_cols = [c for c in result.train_data.columns if c != "default"]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    feature_names = col_info["feature_columns"]
    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": feature_names,
        "default_rate": float(y_train.mean()),
        "test_groups": _decode_groups(
            X_test, feature_names, result.transformers["ordinal_mappings"]
        ),
    }


def to_frame(X: np.ndarray, feature_names: list[str]) -> pl.DataFrame:
    """numpy feature matrix → polars DataFrame with the feature names."""
    return pl.DataFrame(X, schema=feature_names, orient="row")


# ════════════════════════════════════════════════════════════════════════
# BASELINE MODEL — TrainingPipeline (LightGBM) + engine isotonic calibration
# ════════════════════════════════════════════════════════════════════════
# Fixed, reasonable hyperparameters (Exercise 7 showed how to search for
# them) so every technique file trains the identical model. No class
# weighting: Exercise 5 showed it inflates probabilities, and this model's
# probabilities feed conformal sets and a model card.

MODEL_NAME = "credit_default_ex8"
BASELINE_PARAMS: dict[str, Any] = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 5,
    "num_leaves": 31,
    "min_child_samples": 40,
    "random_state": RANDOM_SEED,
    "verbose": -1,
}
_DB_ABS_PATH = (OUTPUT_DIR / "ex8_models.db").resolve()
DB_URL: str = os.environ.get("MLFP03_EX8_DB_URL", f"sqlite:///{_DB_ABS_PATH.as_posix()}")


async def open_registry() -> tuple[Any, Any]:
    """Return ``(ModelRegistry, ConnectionManager)``; caller closes the connection."""
    from kailash.db import ConnectionManager
    from kailash_ml import ModelRegistry

    conn = ConnectionManager(DB_URL)
    await conn.initialize()
    return ModelRegistry(conn), conn


async def _train_and_calibrate(
    X_train: np.ndarray, y_train: np.ndarray, feature_names: list[str]
) -> tuple[Any, int]:
    from kailash_ml import TrainingPipeline
    from kailash_ml.engines.training_pipeline import EvalSpec, ModelSpec

    # 80% of the training rows fit the model; the other 20% (stratified)
    # are kept aside to fit the isotonic calibration map.
    idx_fit, idx_cal = train_test_split(
        np.arange(len(y_train)), test_size=0.2, stratify=y_train, random_state=RANDOM_SEED
    )
    fit_frame = to_frame(X_train[idx_fit], feature_names).with_columns(
        pl.Series("default", y_train[idx_fit]),
        pl.int_range(0, len(idx_fit), dtype=pl.Int64).alias("row_id"),
    )
    schema = FeatureSchema(
        name="ex8_credit_input",
        features=[FeatureField(name=f, dtype="float64") for f in feature_names],
        entity_id_column="row_id",
    )
    registry, conn = await open_registry()
    try:
        pipeline = TrainingPipeline(feature_store=None, registry=registry)
        result = await pipeline.train(
            data=fit_frame,
            schema=schema,
            model_spec=ModelSpec(
                model_class="lightgbm.LGBMClassifier",
                framework="lightgbm",
                hyperparameters=BASELINE_PARAMS,
            ),
            eval_spec=EvalSpec(metrics=["auc"], split_strategy="holdout", test_size=0.2),
            experiment_name=MODEL_NAME,
        )
        if result.model_version is None:
            raise RuntimeError("TrainingPipeline did not register the model")
        version = int(result.model_version.version)
        # Unpickling executes code: only load artefacts you trained yourself.
        base = pickle.loads(await registry.load_artifact(MODEL_NAME, version))
        calibrated = await pipeline.calibrate(
            base,
            to_frame(X_train[idx_cal], feature_names),
            pl.Series("default", y_train[idx_cal]),
            method="isotonic",
        )
    finally:
        await conn.close()
    return calibrated, version


def train_calibrated_model(
    X_train: np.ndarray, y_train: np.ndarray, feature_names: list[str]
) -> Any:
    """Train LightGBM via TrainingPipeline, then isotonic-calibrate it.

    The base model is registered (stage: staging) in this exercise's
    registry as ``MODEL_NAME``; calibration uses TrainingPipeline.calibrate
    on a held-out 20% slice of the training rows.
    """
    calibrated, _ = asyncio.run(_train_and_calibrate(X_train, y_train, feature_names))
    return calibrated


# ════════════════════════════════════════════════════════════════════════
# DECISION THRESHOLD + PRODUCTION ARTEFACTS shared by 8.3 - 8.5
# ════════════════════════════════════════════════════════════════════════
# An applicant is FLAGGED (predicted default — the adverse outcome) when
# p(default) >= DECISION_THRESHOLD. With calibrated probabilities the
# cost-minimising cut-off is c_FP / (c_FP + c_FN) (Exercise 5). The costs
# are ILLUSTRATIVE, the same as Exercise 5: a missed default (FN) costs
# S$10,000, wrongly declining a good applicant (FP) costs S$1,500.
COST_FN_SGD = 10_000.0
COST_FP_SGD = 1_500.0
DECISION_THRESHOLD = COST_FP_SGD / (COST_FP_SGD + COST_FN_SGD)

PRODUCTION_MODEL_NAME = "credit_default_production"  # registered by 8.4
CARD_PATH = OUTPUT_DIR / "ex8_03_model_card.md"  # written by 8.3
CARD_EVIDENCE_PATH = OUTPUT_DIR / "ex8_03_model_card_evidence.json"  # written by 8.3


def evaluate_classification(
    y_true: np.ndarray, y_proba: np.ndarray, threshold: float = 0.5
) -> dict[str, float]:
    """Return the full classification metric bundle used across ex_8."""
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
    }


# ════════════════════════════════════════════════════════════════════════
# SPLIT CONFORMAL PREDICTION (binary classification)
# ════════════════════════════════════════════════════════════════════════
# Score s = 1 - p(true class | x). With n calibration scores, q̂ is the
# ⌈(n+1)(1-α)⌉-th smallest score — np.quantile at level ⌈(n+1)(1-α)⌉/n
# with method="higher" picks exactly that order statistic (the default
# linear interpolation can land slightly below it and void the guarantee).
# Guarantee: P(Y ∈ C(X)) ≥ 1-α on AVERAGE over future applicants
# (marginal coverage), assuming calibration and future data are
# exchangeable. It is not a per-applicant promise.


def nonconformity_scores(y_true: np.ndarray, p_default: np.ndarray) -> np.ndarray:
    """1 - probability the model gave to the class that actually happened."""
    return np.where(y_true == 1, 1.0 - p_default, p_default)


def conformal_qhat(cal_scores: np.ndarray, alpha: float) -> float:
    """Finite-sample-corrected conformal threshold q̂."""
    n = len(cal_scores)
    level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return float(np.quantile(cal_scores, level, method="higher"))


def prediction_sets(p_default: np.ndarray, q_hat: float) -> tuple[np.ndarray, np.ndarray]:
    """Boolean membership of class 0 and class 1 in each prediction set.

    A set may be EMPTY (neither class plausible at this α) — that is a
    legitimate conformal outcome, counted as a miss for coverage.
    """
    has_1 = (1.0 - p_default) <= q_hat
    has_0 = p_default <= q_hat
    return has_0, has_1


def conformal_summary(
    y_true: np.ndarray, p_default: np.ndarray, q_hat: float
) -> dict[str, float]:
    """Coverage, average set size and singleton / empty / both-class rates."""
    has_0, has_1 = prediction_sets(p_default, q_hat)
    covered = np.where(y_true == 1, has_1, has_0)
    size = has_0.astype(int) + has_1.astype(int)
    return {
        "coverage": float(covered.mean()),
        "avg_set_size": float(size.mean()),
        "singleton_rate": float((size == 1).mean()),
        "both_rate": float((size == 2).mean()),
        "empty_rate": float((size == 0).mean()),
    }


def conformal_on_test(
    y_test: np.ndarray, p_test: np.ndarray, alpha: float = 0.10
) -> dict[str, float]:
    """8.1's protocol in one call: q̂ from the first half of the test rows,
    coverage and set sizes measured on the second half."""
    n_cal = len(y_test) // 2
    q_hat = conformal_qhat(nonconformity_scores(y_test[:n_cal], p_test[:n_cal]), alpha)
    return {"alpha": alpha, "q_hat": q_hat, **conformal_summary(y_test[n_cal:], p_test[n_cal:], q_hat)}


# ════════════════════════════════════════════════════════════════════════
# FAIRNESS — measured per protected group (used by the model card + gate)
# ════════════════════════════════════════════════════════════════════════


def fairness_report(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    groups: dict[str, np.ndarray],
    threshold: float,
) -> pl.DataFrame:
    """Per-group rates for each protected attribute.

    "Flagged" means predicted default (y_proba >= threshold) — the
    adverse outcome for an applicant. Columns: attribute, group, n,
    base_rate, flag_rate, tpr, fpr, mean_pred.
    """
    flagged = y_proba >= threshold
    rows = []
    for attr, labels in groups.items():
        for g in np.unique(labels):
            m = labels == g
            pos, neg = m & (y_true == 1), m & (y_true == 0)
            rows.append(
                {
                    "attribute": attr,
                    "group": str(g),
                    "n": int(m.sum()),
                    "base_rate": float(y_true[m].mean()),
                    "flag_rate": float(flagged[m].mean()),
                    "tpr": float(flagged[pos].mean()) if pos.any() else float("nan"),
                    "fpr": float(flagged[neg].mean()) if neg.any() else float("nan"),
                    "mean_pred": float(y_proba[m].mean()),
                }
            )
    return pl.DataFrame(rows)


def fairness_summary(report: pl.DataFrame, min_group_n: int = 200) -> pl.DataFrame:
    """Per attribute: disparate-impact ratio and equalised-odds gaps.

    Disparate impact here = lowest group flag rate / highest group flag
    rate (1.0 = parity; the four-fifths rule of thumb flags < 0.8).
    Groups smaller than ``min_group_n`` are excluded as too noisy.
    """
    big = report.filter(pl.col("n") >= min_group_n)
    return (
        big.group_by("attribute")
        .agg(
            (pl.col("flag_rate").min() / pl.col("flag_rate").max()).alias("disparate_impact"),
            (pl.col("tpr").max() - pl.col("tpr").min()).alias("tpr_gap"),
            (pl.col("fpr").max() - pl.col("fpr").min()).alias("fpr_gap"),
            (pl.col("mean_pred") - pl.col("base_rate")).abs().max().alias("max_calibration_gap"),
            pl.len().alias("groups"),
        )
        .sort("attribute")
    )


# ════════════════════════════════════════════════════════════════════════
# DRIFT STATISTICS — PSI + KS (the maths DriftMonitor runs for you)
# ════════════════════════════════════════════════════════════════════════
#
# PSI (Population Stability Index):
#     PSI = Σ (p_new - p_ref) * ln(p_new / p_ref)
#     common rule of thumb: < 0.1 no shift, 0.1-0.2 moderate, > 0.2 significant
# KS (Kolmogorov-Smirnov): two-sample test on the empirical CDFs.

# Continuous features used for drift simulations. (The first columns of
# the matrix are ordinal-encoded categoricals such as gender — shifting
# those by "0.5 standard deviations" would be meaningless.)
DRIFT_FEATURES: list[str] = ["income_sgd", "debt_to_income", "credit_utilization"]


def compute_psi(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index on 1-D arrays.

    Bin edges come from the reference; the outer edges are opened to ±inf
    so current values outside the reference range still count (dropping
    them would understate exactly the shifts PSI exists to catch).
    """
    _, edges = np.histogram(reference, bins=bins)
    edges = edges.astype(float)
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(reference, bins=edges)
    cur_counts, _ = np.histogram(current, bins=edges)
    # Laplace-smooth to avoid log(0)
    ref_props = (ref_counts + 1) / (len(reference) + bins)
    cur_props = (cur_counts + 1) / (len(current) + bins)
    return float(np.sum((cur_props - ref_props) * np.log(cur_props / ref_props)))


def compute_ks(reference: np.ndarray, current: np.ndarray) -> tuple[float, float]:
    """Return (KS statistic, p-value) on 1-D arrays."""
    ks_stat, p_value = stats.ks_2samp(reference, current)
    return float(ks_stat), float(p_value)


def drift_row(
    reference: np.ndarray, current: np.ndarray, psi_threshold: float = 0.1
) -> dict[str, float | str]:
    """Return {psi, ks_stat, ks_pval, drift} for a single feature."""
    psi = compute_psi(reference, current)
    ks_stat, ks_pval = compute_ks(reference, current)
    drift = "YES" if (psi > psi_threshold or ks_pval < 0.05) else "No"
    return {"psi": psi, "ks_stat": ks_stat, "ks_pval": ks_pval, "drift": drift}


def simulate_gradual_drift(
    X_ref: np.ndarray, X_new: np.ndarray, feature_idx: list[int], shift: float = 0.5
) -> np.ndarray:
    """Copy of X_new with the given columns mean-shifted by ``shift`` σ."""
    drifted = X_new.copy()
    for i in feature_idx:
        drifted[:, i] += shift * X_ref[:, i].std()
    return drifted


def simulate_sudden_drift(
    X_ref: np.ndarray,
    X_new: np.ndarray,
    feature_idx: int,
    sigma_shift: float = 3.0,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Replace one column in X_new with a heavily shifted Gaussian."""
    rng = np.random.default_rng(seed)
    drifted = X_new.copy()
    drifted[:, feature_idx] = rng.normal(
        loc=X_ref[:, feature_idx].mean() + sigma_shift * X_ref[:, feature_idx].std(),
        scale=X_ref[:, feature_idx].std(),
        size=drifted.shape[0],
    )
    return drifted




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 8.1: Conformal Prediction for Credit Default
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Train a calibrated LightGBM credit-default model through TrainingPipeline
#   - Apply split conformal prediction for distribution-free uncertainty
#   - Check the 1-α (marginal) coverage guarantee on held-out data
#   - Sweep α and read the cost of tighter coverage in set size
#   - Translate "ambiguous prediction set" into a business routing rule
#
# PREREQUISITES: MLFP03 Exercises 1-7, MLFP02 (preprocessing).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory     — what conformal guarantees, and under which assumption
#   2. Build      — train + calibrate the model, compute nonconformity scores
#   3. Train      — calibrate q̂, generate prediction sets, measure coverage
#   4. Visualise  — plot coverage + set-size mix across α values
#   5. Apply      — which applicants a Singapore lender sends to human review
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go



## THEORY — What conformal prediction guarantees


In [ ]:
# Every classifier outputs a probability, but a probability is not a
# guarantee. A model that says "80% chance of default" on 100 applicants
# may see 40 defaults (over-confident) or 95 (under-confident).
#
# Conformal prediction asks a different question: "Which SET of labels
# C(x) should I report so that the true label is inside it for at least
# 1 - α of applicants?"
#
# The one assumption is EXCHANGEABILITY: calibration applicants and
# future applicants are drawn from the same distribution, in no
# particular order. No Gaussian assumption and no requirement that the
# model is accurate or calibrated — a bad model simply produces bigger
# (less useful) sets.
#
# The guarantee is MARGINAL: averaged over future applicants,
# P(Y ∈ C(X)) ≥ 1 - α. It does not promise 1 - α for any single
# applicant or for every sub-group.
#
# THE ALGORITHM (split conformal, binary classification):
#   1. Split held-out data into CALIBRATION and EVALUATION halves.
#   2. On CALIBRATION: nonconformity score s_i = 1 - p(y_true | x_i).
#      High score = the model was surprised by what happened.
#   3. q̂ = the ⌈(n+1)(1-α)⌉-th smallest score (finite-sample correction).
#   4. On new x: include class c in the set iff 1 - p(c|x) ≤ q̂.
#   5. Sets of size 2 mean "both outcomes plausible"; an empty set means
#      "neither is plausible at this α" — rare, and counted as a miss.
#
# THE BUSINESS PAYOFF: each prediction comes with a routing decision.
#   - Singleton {0} or {1}: one outcome is plausible → decide automatically
#   - {0, 1}: the model cannot separate the outcomes → route to a human



## TASK 2 — BUILD: train the calibrated baseline model


In [ ]:
print("\n" + "=" * 70)
print("  MLFP03 Exercise 8.1 — Conformal Prediction")
print("=" * 70)

split = load_credit_split()
X_train, y_train = split["X_train"], split["y_train"]
X_test, y_test = split["X_test"], split["y_test"]
feature_names = split["feature_names"]

print(f"\nData: train={X_train.shape}, test={X_test.shape}")
print(f"Default rate: {split['default_rate']:.1%}")

# TODO: train + isotonic-calibrate the baseline with the shared helper
# Hint: train_calibrated_model(<X>, <y>, <feature names>)
calibrated_model = ____
y_proba = calibrated_model.predict_proba(X_test)[:, 1]
metrics = evaluate_classification(y_test, y_proba)

print("\n=== Calibrated Model Metrics (test) ===")
for k, v in metrics.items():
    print(f"  {k:<10} {v:.4f}")
no_skill_brier = split["default_rate"] * (1 - split["default_rate"])
print(f"  (Brier of always predicting the base rate: {no_skill_brier:.4f})")



### Checkpoint 1


In [ ]:
assert metrics["auc_roc"] > 0.5, "Task 2: Should beat random"
assert (
    0 < metrics["brier"] < no_skill_brier
), "Task 2: Brier should beat the constant base-rate forecast"
print("\n[ok] Checkpoint 1 — calibrated LightGBM trained and evaluated\n")



## TASK 3 — TRAIN the conformal calibrator


In [ ]:
# Split the test set into calibration and evaluation halves. Compute
# nonconformity scores on calibration, then apply q̂ on evaluation.

n_cal = X_test.shape[0] // 2
X_cal, X_eval = X_test[:n_cal], X_test[n_cal:]
y_cal, y_eval = y_test[:n_cal], y_test[n_cal:]

cal_proba = calibrated_model.predict_proba(X_cal)[:, 1]
# TODO: nonconformity score = 1 - probability given to the TRUE class
# Hint: nonconformity_scores(<true labels>, <p(default)>)
cal_scores = ____

alpha = 0.10  # target 90% coverage
# TODO: finite-sample conformal threshold for this α
# Hint: conformal_qhat(<calibration scores>, <alpha>)
q_hat = ____

print("=== Conformal Calibration ===")
print(f"  Calibration set:   {len(cal_scores)} samples")
print(f"  Target coverage:   {1 - alpha:.0%}")
print(f"  Calibration q̂:     {q_hat:.4f}")

eval_proba = calibrated_model.predict_proba(X_eval)[:, 1]
# TODO: which classes enter each evaluation applicant's set?
# Hint: prediction_sets(<p(default)>, <q̂>) returns (has_0, has_1)
has_0, has_1 = ____
summary = conformal_summary(y_eval, eval_proba, q_hat)
coverage = summary["coverage"]
singleton_rate = summary["singleton_rate"]

print("\n=== Empirical Results on Evaluation Half ===")
print(f"  Coverage:          {coverage:.4f} (target: ≥ {1 - alpha:.2f})")
print(f"  Avg set size:      {summary['avg_set_size']:.3f}")
print(f"  Singleton rate:    {singleton_rate:.1%} (auto-decide)")
print(f"  Both-class rate:   {summary['both_rate']:.1%} (route to human)")
print(f"  Empty-set rate:    {summary['empty_rate']:.1%}")



### Checkpoint 2


In [ ]:
# Coverage on a finite evaluation half fluctuates around its expectation;
# a 2-point tolerance absorbs that sampling noise.
assert coverage >= (
    1 - alpha - 0.02
), f"Task 3: Coverage {coverage:.4f} should be near target {1 - alpha:.4f}"
assert 0 <= summary["avg_set_size"] <= 2, "Task 3: Set size must be in [0, 2]"
# INTERPRETATION: coverage holds while new applicants look like the
# calibration applicants. If live coverage drops below target, the data
# has shifted: the first remedy is to RECALIBRATE q̂ on recent labelled
# data (cheap); retrain the model if its ranking (AUC) has degraded too.
print("\n[ok] Checkpoint 2 — conformal coverage verified on held-out data\n")



## TASK 4 — VISUALISE coverage vs α


In [ ]:
# Lower α → higher coverage but bigger sets (more human review).

print("=== Coverage vs α Sweep ===")
print(f"{'Alpha':>8} {'Target':>10} {'Actual':>10} {'Avg Size':>10} {'Singleton':>12}")
print("─" * 54)

alphas_sweep = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
sweep_rows = []
for a in alphas_sweep:
    # TODO: summarise coverage / set sizes at this α (recompute q̂ for `a`)
    # Hint: conformal_summary(<y>, <p>, <q̂ for a>)
    row = ____
    sweep_rows.append({"alpha": a, **row})
    print(
        f"{a:>8.2f} {1 - a:>10.2f} {row['coverage']:>10.4f} "
        f"{row['avg_set_size']:>10.3f} {row['singleton_rate']:>11.1%}"
    )

xs = [r["alpha"] for r in sweep_rows]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=xs,
        y=[r["coverage"] for r in sweep_rows],
        mode="lines+markers",
        name="Empirical coverage",
        line=dict(color="#2563eb", width=3),
    )
)
fig.add_trace(
    go.Scatter(
        x=xs,
        y=[1 - a for a in xs],
        mode="lines",
        name="Target 1-α",
        line=dict(color="#94a3b8", dash="dash"),
    )
)
fig.add_trace(
    go.Bar(
        x=xs,
        y=[r["singleton_rate"] for r in sweep_rows],
        name="Singleton rate (auto-decide)",
        marker_color="#10b981",
        opacity=0.55,
        yaxis="y2",
    )
)
fig.update_layout(
    title="Conformal Prediction: Coverage vs α (Singapore credit default)",
    xaxis_title="α (risk budget)",
    yaxis=dict(title="Coverage", range=[0, 1.05]),
    yaxis2=dict(title="Singleton rate", overlaying="y", side="right", range=[0, 1.05]),
    legend=dict(orientation="h", y=-0.2),
    height=500,
)
viz_path = OUTPUT_DIR / "ex8_01_conformal_sweep.html"
fig.write_html(str(viz_path))
print(f"\nSaved: {viz_path}")



### Checkpoint 3


In [ ]:
assert len(sweep_rows) == len(alphas_sweep), "Task 4: Should sweep all alphas"
for r in sweep_rows:
    assert r["coverage"] >= 1 - r["alpha"] - 0.02, (
        f"Task 4: coverage at α={r['alpha']} fell below target"
    )
print("\n[ok] Checkpoint 3 — α sweep visualised\n")



## TASK 5 — APPLY: routing applications at a Singapore lender


In [ ]:
# SCENARIO (illustrative): a Singapore retail lender receives ~18,000
# unsecured-loan applications a month and today has an analyst review
# every one. Conformal sets give a routing rule:
#
#   {0}     → fast-track approval (only "no default" is plausible)
#   {1}     → decline / strict review (only "default" is plausible)
#   {0, 1}  → HUMAN REVIEW (the model cannot separate the outcomes)
#
# What the 90% guarantee does and does not say: across all applicants,
# at least 90% of sets contain the true outcome — provided new
# applicants look like the calibration applicants. It is NOT "90% sure
# about this applicant", and a recession that changes who applies breaks
# the exchangeability assumption (8.2 watches for that).

# TODO: applicants whose set is exactly {0}
# Hint: combine the boolean arrays with & and ~
only_0 = ____
only_1 = has_1 & ~has_0
both = has_0 & has_1
print(f"=== Routing at α={alpha} on {len(y_eval):,} held-out applicants ===")
for name, mask in (("{0} fast-track", only_0), ("{1} decline/strict", only_1), ("{0,1} human review", both)):
    rate = float(mask.mean())
    actual = f"{y_eval[mask].mean():.1%}" if mask.any() else "n/a (empty)"
    print(f"  {name:<20} {rate:>6.1%} of applicants   observed default rate {actual}")

# DOLLAR IMPACT — ILLUSTRATIVE assumptions, not any lender's figures
monthly_apps = 18_000
analyst_hourly = 85.0  # SGD, fully loaded
time_per_review_h = 0.25  # 15 minutes per application
review_share = float(both.mean() + only_1.mean())  # humans still check declines
status_quo_cost = monthly_apps * time_per_review_h * analyst_hourly
conformal_cost = monthly_apps * review_share * time_per_review_h * analyst_hourly
savings = status_quo_cost - conformal_cost
print(f"\n  Analyst cost (status quo, 100% review): S${status_quo_cost:>10,.0f}/mo")
print(f"  Analyst cost (review {{0,1}} + {{1}} only): S${conformal_cost:>10,.0f}/mo")
print(f"  Monthly savings:                         S${savings:>10,.0f}/mo")
print(f"  Annualised:                              S${savings * 12:>10,.0f}/yr")
# INTERPRETATION: the fast-tracked {0} group is only safe if its observed
# default rate (printed above) is acceptable to the credit policy —
# check that number, not just the share of work saved.

# LIMITATIONS:
#   - Coverage is marginal; it can be lower inside a sub-group. Check
#     group-wise coverage alongside the fairness audit (Exercise 6, 8.3).
#   - Exchangeability breaks under drift; 8.2 monitors for it.



## REFLECTION


[x] Trained and calibrated the credit model through TrainingPipeline
  [x] Built split conformal prediction from nonconformity scores
  [x] Measured {coverage:.1%} coverage at α={alpha} on held-out applicants
      (a marginal guarantee under exchangeability)
  [x] Swept α and visualised the cost of tighter coverage
  [x] Turned set size into a routing rule worth
      ~S${savings * 12:,.0f}/yr of analyst time (illustrative inputs)

  KEY INSIGHT: A probability is a guess. A prediction set comes with a
  coverage guarantee — on average, and only while tomorrow's applicants
  resemble the calibration set.

  Next: 02_drift_monitoring.py — detect when that resemblance breaks.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

